In [1]:
import torch
import torch.nn as nn
import tiktoken
from common.LLMUtils import *

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "ebd_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

In [3]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x

class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-3):
        super().__init__()

    def forward(self, x):
        return x

In [4]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["ebd_dim"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["ebd_dim"])
        self.dropout_embedding = nn.Dropout(cfg["drop_rate"])

        self.transformer_layers = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])],
        )
        self.finalNorm = DummyLayerNorm(cfg["ebd_dim"])
        self.out_head = nn.Linear(cfg["ebd_dim"], cfg["vocab_size"], bias=False)


    def forward(self, x):
        batch_size, seq_len = x.shape

        tokens_ebd = self.token_embedding(x)
        pos_ebd = self.position_embedding(torch.arange(0, seq_len, device=tokens_ebd.device))

        res = tokens_ebd + pos_ebd
        res = self.dropout_embedding(res)
        res = self.transformer_layers(res)
        res = self.finalNorm(res)

        logit = self.out_head(res)
        return logit




In [5]:
tokenizer = tiktoken.get_encoding('gpt2')
text1 = "Every effort moves you"
text2 = "Every day holds a"
batch = []
batch.append(torch.tensor(tokenizer.encode(text1)))
batch.append(torch.tensor(tokenizer.encode(text2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [6]:
modal = DummyGPTModel(GPT_CONFIG_124M)
logits = modal(batch)
print(logits)

tensor([[[ 0.1304,  0.7486,  0.3687,  ..., -0.1247, -0.9480,  1.0912],
         [-0.8022,  0.2327, -0.2428,  ..., -0.1582, -0.3011, -0.3510],
         [-0.8898,  0.2675,  0.1318,  ..., -1.2451, -0.8092, -1.0382],
         [ 0.7204,  0.5181, -0.3175,  ..., -0.2824,  1.7130,  0.2879]],

        [[-1.2592,  0.9664,  0.3366,  ...,  0.2273, -0.5572,  0.6316],
         [-1.7606, -0.8263, -0.1862,  ..., -0.4803,  1.0272, -1.0510],
         [ 0.2147,  0.5484, -1.6459,  ..., -1.3665,  0.4876, -0.6743],
         [-0.1354,  0.6026, -0.4807,  ..., -0.7401,  1.2674, -0.2935]]],
       grad_fn=<UnsafeViewBackward0>)


In [7]:
test_batch = torch.randn(2, 5)
layer = nn.Sequential(
    nn.Linear(5, 6),
    nn.ReLU(),
)
out = layer(test_batch)
print(out)

tensor([[0.0371, 0.9331, 0.0000, 0.9862, 0.0000, 0.0000],
        [0.2038, 0.0216, 0.0000, 0.2454, 0.4472, 0.0000]],
       grad_fn=<ReluBackward0>)


In [8]:
mean = out.mean(dim=-1)
print(mean)
varince = out.var(dim=-1)

tensor([0.3261, 0.1530], grad_fn=<MeanBackward1>)


In [9]:
torch.set_printoptions(sci_mode=False)
test_batch = torch.randn(2, 5)
norm_layer = LayerNorm(ebd_dim=5)
out = norm_layer(test_batch)
print(out)
print(out.mean(dim=-1, keepdim=True))
print(out.var(dim=-1, unbiased=False, keepdim=True))

tensor([[-0.0490,  0.8793, -0.4947,  1.2328, -1.5684],
        [-0.6027,  0.1551,  0.2246, -1.3947,  1.6177]], grad_fn=<AddBackward0>)
tensor([[0.],
        [0.]], grad_fn=<MeanBackward1>)
tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [10]:
x = torch.randn(2, 4, 768)
transformer_block = TransformerBlock(GPT_CONFIG_124M)
out = transformer_block(x)
print(x.shape)
print(out.shape)

torch.Size([2, 4, 768])
torch.Size([2, 4, 768])
